# Проверка экспериментальных групп

Проверю, соответствуют ли размеры трёх групп ожидаемому распределению 1:1:1 и насколько группы сопоставимы по характеристикам клиентов до рассылки.

Сначала посчитаю участников в каждой группе, затем выполню SRM-проверку и сравню исторические расходы, давность последней покупки и другие признаки. Эффект рассылок оценю в следующем ноутбуке.

In [3]:
from pathlib import Path

import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

df = pd.read_csv(root / "data" / "hillstrom.csv")
print(f"Загружено строк: {len(df):,}")

Загружено строк: 64,000


In [4]:
# посчитаем участников в каждой группе 
group_counts = df["segment"].value_counts(dropna=False)

group_summary = group_counts.to_frame("Клиентов")
group_summary["Доля, %"] = (group_counts / len(df) * 100).round(2)

display(group_summary)

,Клиентов,"Доля, %"
segment,,
Womens E-Mail,21387,33.42
Mens E-Mail,21307,33.29
No E-Mail,21306,33.29


### Проверка размеров групп (SRM)

По дизайну эксперимента у клиента была одинаковая вероятность попасть в каждую из трёх групп. Проверю, согласуются ли наблюдаемые размеры групп с пропорцией 1:1:1, с помощью критерия χ² Пирсона. Порог для диагностической проверки — 0,001.

In [ ]:
from scipy.stats import chisquare

expected_counts = [len(df) / 3] * 3
alpha_srm = 0.001

statistic, p_value = chisquare(
    f_obs=group_counts.to_numpy(),
    f_exp=expected_counts,
)

print(f"χ²: {statistic:.4f}")
print(f"p-value: {p_value:.4f}")

if p_value < alpha_srm:
    print("Обнаружен сигнал SRM.")
else:
    print("Сигнал SRM не обнаружен.")

χ²: 0.2025
p-value: 0.9037
Сигнал SRM не обнаружен.


### Сравнение групп до рассылки

Сравню каждую группу рассылки с контролем по давности последней покупки и историческим расходам. Посчитаю средние, медианы и стандартизированную разницу средних (SMD).

In [6]:
numeric_features = ["recency", "history"]

baseline_summary = (
    df.groupby("segment")[numeric_features]
    .agg(["mean", "median", "std"])
)

display(baseline_summary.round(2))

recency              history                
                 mean median   std    mean  median     std
segment                                                   
Mens E-Mail      5.77    6.0  3.51  242.84  157.22  260.36
No E-Mail        5.75    5.0  3.50  240.88  156.66  252.74
Womens E-Mail    5.77    6.0  3.51  242.54  160.09  255.33

SMD показывает, насколько далеко друг от друга средние групп по сравнению с разбросом самого признака. Например, SMD = 0,02 означает, что разница средних составляет всего 0,02 стандартного отклонения.

In [ ]:
# SMD для каждой рассылки
control = df.loc[df["segment"] == "No E-Mail", numeric_features]
smd_by_group = {}

for group in ["Mens E-Mail", "Womens E-Mail"]:
    treatment = df.loc[df["segment"] == group, numeric_features]

    mean_diff = treatment.mean() - control.mean()
    pooled_std = ((treatment.var() + control.var()) / 2) ** 0.5

    smd_by_group[group] = mean_diff / pooled_std

display(pd.DataFrame(smd_by_group).round(4))

,Mens E-Mail,Womens E-Mail
recency,0.0068,0.0052
history,0.0076,0.0065


По давности последней покупки и расходам за предыдущий год средние значения в обеих группах рассылки близки к контрольной группе. Максимальная стандартизированная разница средних составляет 0,0076.

In [8]:
binary_features = ["newbie", "mens", "womens"]
categorical_features = ["channel", "zip_code", "history_segment"]

category_indicators = pd.get_dummies(
    df[categorical_features].fillna("Пропуск"),
    dtype=int,
)

baseline_indicators = pd.concat(
    [df[binary_features], category_indicators],
    axis=1,
)

shares = baseline_indicators.groupby(df["segment"]).mean().T * 100
control_share = shares["No E-Mail"]

balance = shares[["No E-Mail", "Mens E-Mail", "Womens E-Mail"]].copy()
balance["Мужская рассылка − контроль, п.п."] = (
    shares["Mens E-Mail"] - control_share
)
balance["Женская рассылка − контроль, п.п."] = (
    shares["Womens E-Mail"] - control_share
)

display(balance.round(2))

segment,No E-Mail,Mens E-Mail,Womens E-Mail,"Мужская рассылка − контроль, п.п.","Женская рассылка − контроль, п.п."
newbie,50.20,50.15,50.32,-0.04,0.13
mens,55.32,55.09,54.89,-0.23,-0.43
womens,54.76,55.14,55.01,0.38,0.25
channel_Multichannel,12.23,12.09,12.06,-0.14,-0.17
channel_Phone,43.78,43.37,44.20,-0.41,0.43
channel_Web,43.99,44.54,43.74,0.55,-0.26
zip_code_Rural,14.73,15.22,14.87,0.49,0.14
zip_code_Surburban,45.18,44.59,45.12,-0.58,-0.05
zip_code_Urban,40.09,40.19,40.01,0.10,-0.09
history_segment_1) $0 - $100,35.73,36.25,35.69,0.52,-0.03


Наибольшее отличие от контроля — в группе клиентов, которые до эксперимента потратили от $100 до $200. В контроле их 22,70%, в группе мужской рассылки — 22,02% (−0,68 п.п.), в группе женской — 22,10% (−0,60 п.п.). По остальным проверенным признакам разницы ещё меньше.

### Итог проверки экспериментальных групп

В эксперименте участвовали 64 000 клиентов. Проверка размеров групп относительно пропорции 1:1:1 не выявила сигнала SRM (χ² = 0,2025, p-value = 0,9037).

По давности последней покупки и историческим расходам максимальная стандартизированная разница средних составила 0,0076. Максимальная разница в долях по остальным проверенным признакам составила 0,68 процентного пункта.

По проверенным характеристикам до рассылки обе группы близки к контролю. Далее оценю эффект каждой рассылки на средние расходы клиента за две недели.